# Live Reinforcement Learning — critic-free PPO

The animals keep learning **as they live**. While awake, every individual of every species
collects on-policy experience; when the population falls asleep at night the simulation
**pauses** and a PPO update runs; on waking, collection resumes. There is **no `RuleBrain`** in
the loop — both species are driven by their own learning policy, warm-started from the
imitation-learning clones (`../imitation_learning/sheep.pt`, `fox.pt`).

**Reward** (see `RewardConfig` in `ppo_live.py`): eating/hunting, drinking, breeding, and simply
surviving earn reward; an **avoidable** death (starvation, thirst, predation, injury) delivers
*pain*. Dying of **old age is unavoidable, so it carries no pain** — detected heuristically from
the animal's age vs its `max_age` gene at the moment it acted.

**Critic-free PPO.** There is no value network. The advantage of each step is its discounted
return-to-go along that agent's own `birth_id` trajectory, whitened to a per-species baseline
and optimised with the clipped PPO surrogate (plus an entropy bonus and a KL trust region). The
deployed brain therefore stays *actor-only* — the trained policy re-exports as a drop-in
`(head_mean, gate_logits, speed_logit)` TorchScript archive for `run_experiment.py` /
`run_live.py`.

All the machinery lives in **`ppo_live.py`** beside this notebook; the notebook is the driver.

In [ ]:
# --- bootstrap: reach ppo_live.py (this folder) + the imitation toolkit + the repo root ---
import sys, time
from pathlib import Path

_HERE = Path.cwd()
for _c in (_HERE, *_HERE.parents):
    if (_c / "notebooks" / "live_learning" / "ppo_live.py").exists():
        _LIVE = _c / "notebooks" / "live_learning"
        break
    if (_c / "ppo_live.py").exists():
        _LIVE = _c
        break
else:
    raise RuntimeError("run this notebook from inside the ecosystem repo")
if str(_LIVE) not in sys.path:
    sys.path.insert(0, str(_LIVE))
_NB = _LIVE.parent                            # notebooks/ -- holds the shared common.py
if str(_NB) not in sys.path:
    sys.path.insert(0, str(_NB))

import numpy as np
import torch
import matplotlib.pyplot as plt

import ppo_live as P            # the PPO engine (imports common.py -> puts repo root on sys.path)
import common as C
from config import make_config, SHEEP, FOX, SPECIES_NAMES
from sim.simulation import Simulation

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# --- configuration ---------------------------------------------------------------------
WORLD_SEED = 12345          # fixes the map (terrain + hydrology)
RUN_SEED   = 7              # fixes the sim dynamics RNG (action sampling adds its own noise)
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# how long to train.
#   An "epoch" == one full episode on the SAME world + run seed: the Simulation is rebuilt from
#     byte-identical initial conditions (make_config with the fixed WORLD_SEED/RUN_SEED), run
#     until the whole population dies out or N_CYCLES day/night cycles elapse, then reset and run
#     again. Only the policies + optimizers (and torch's action-sampling RNG) carry across
#     epochs, so both species keep improving on repeated runs of the identical scenario. This is
#     what lets training survive an extinction: the sim resets instead of the notebook stopping.
#   A "cycle"  == one day of collection + one night PPO update (~1 in-game day, ~240 ticks).
N_EPOCHS   = 30             # number of episodes (sim resets) to run; raise for a longer run
N_CYCLES   = 40             # max day/night cycles PER EPOCH before the sim is reset
MAX_TICKS  = 200_000        # per-epoch safety cap on sim ticks (guards a day that never ends)
SAVE_EVERY = 5              # export deployable checkpoints every N cycles (counted across epochs)

# warm start from the imitation clones (set to None to learn from scratch)
WARM_SHEEP = C.MODEL_PATHS[SHEEP]
WARM_FOX   = C.MODEL_PATHS[FOX]

# where the fine-tuned, deployable policies are written (kept separate from the clones)
OUT_SHEEP = _LIVE / "sheep_ppo.pt"
OUT_FOX   = _LIVE / "fox_ppo.pt"

ppocfg = P.PPOConfig(
    gamma=0.99, clip=0.2, epochs=4, minibatch=1024, lr=1e-4,
    entropy_coef=0.005, max_grad_norm=0.5, target_kl=0.03,
    max_agents=128,        # trajectories tracked per species per day (bounds memory)
    night_hi=0.5,          # >= this fraction asleep  -> pause + train
    night_lo=0.2,          # <= this fraction asleep  -> daytime, resume collecting
    min_cycle=60,          # min ticks collected before a night trigger is allowed
    horizon=400,           # force a train if a day runs longer than this (safety)
)
rcfg = P.RewardConfig()    # reward/pain weights -- tweak freely

torch.manual_seed(RUN_SEED)   # reproducible action sampling (independent of the sim's numpy RNG)
ppocfg, rcfg

In [ ]:
# --- build the two learning policies (warm-started) + a single brain that drives both ------
# The policies, optimizers, buffers and brain are created ONCE here and reused across every
# epoch -- that is how both species carry what they learned from one episode into the next. The
# Simulation itself is rebuilt inside the epoch loop below (from the same WORLD_SEED/RUN_SEED),
# and Simulation.__init__ re-binds this same `brain` to the fresh entities each epoch.
policies = {
    SHEEP: P.build_ppo_policy(SHEEP, warm_start=WARM_SHEEP, device=DEVICE),
    FOX:   P.build_ppo_policy(FOX,   warm_start=WARM_FOX,   device=DEVICE),
}
optimizers = {sid: torch.optim.Adam(policies[sid].parameters(), lr=ppocfg.lr)
              for sid in (SHEEP, FOX)}
buffers = {sid: P.RolloutBuffer(gamma=ppocfg.gamma, max_agents=ppocfg.max_agents)
           for sid in (SHEEP, FOX)}

brain = P.LivePPOBrain(policies, device=DEVICE)   # no RuleBrain, no CompositeBrain
brain.training = True

## The live training loop

**Outer loop — epochs (episodes).** Each epoch rebuilds the `Simulation` from the *same*
`WORLD_SEED` + `RUN_SEED`, so every epoch replays the byte-identical initial world. The
persistent `policies`/`optimizers`/`buffers` (and torch's action-sampling RNG) carry across
epochs, so both species keep improving on repeated runs of the identical scenario. An epoch
ends when the whole population dies out, after `N_CYCLES` day/night cycles, or at the
`MAX_TICKS` safety cap — then the sim resets and the next epoch begins (instead of the notebook
stopping on extinction).

**Inner loop — cycles (days).** A small state machine follows the sun (the sim starts at night,
`time_of_day ≈ 0`):

- **Not collecting** → wait for daybreak (`asleep_frac ≤ night_lo`), then start a new day of
  collection.
- **Collecting** → each tick, snapshot state → `sim.step()` (the brain samples actions and
  records them) → reward by snapshot-diff → file into the per-species buffer. When most animals
  bed down (`asleep_frac ≥ night_hi`, after at least `min_cycle` ticks) — or a safety `horizon`
  is hit — **pause and run the PPO update**, then clear the buffers and sleep through the night.

Sleeping steps aren't collected at all; the occasional dusk straggler whose action the sleep
system overrides is dropped from the policy gradient via the `controlled` mask.

In [ ]:
# --- run it: OUTER epoch loop over episodes of the SAME world ----------------------------
history = []       # per-cycle metrics across ALL epochs (each rec carries its 'epoch')
pop_log = []       # (epoch, tick, n_sheep, n_fox) every tick, for the population plot
global_cycle = 0   # cumulative completed cycles across epochs (drives SAVE_EVERY + plot x-axis)
t0 = time.time()
sim = None

for epoch in range(N_EPOCHS):
    # rebuild the sim from the SAME world + run seed -> byte-identical initial conditions each
    # epoch. Simulation.__init__ re-binds the persistent `brain` (both policies) to the fresh
    # entities; the buffers are cleared so no experience leaks across an episode boundary.
    cfg = make_config(world_seed=WORLD_SEED, seed=RUN_SEED)
    sim = Simulation(cfg, brain=brain)                # the brain instance drives ALL species
    ent = sim.entities
    for _b in buffers.values():
        _b.clear()
    epoch_start_tick = sim.tick
    collecting = False
    cycle = 0                            # per-epoch cycle index
    cycle_start = sim.tick
    print(f"\n=== epoch {epoch:3d}/{N_EPOCHS} | seeded: {sim.populations} ===")

    while cycle < N_CYCLES and (sim.tick - epoch_start_tick) < MAX_TICKS:
        snap = P.snapshot(ent)
        brain.collecting = collecting
        stats = sim.step()
        pop_log.append((epoch, sim.tick, stats["n_sheep"], stats["n_fox"]))

        n_alive = ent.n_alive
        if n_alive == 0:
            print(f"  population went extinct @ tick {sim.tick} — ending epoch {epoch}")
            break

        # reward every agent that acted this tick and file it into its species buffer
        for sid, pend in brain.pending.items():
            r, done, controlled = P.compute_rewards(rcfg, snap, ent, pend)
            buffers[sid].add(pend, r, done, controlled)

        asleep_frac = stats["n_asleep"] / max(1, n_alive)

        if not collecting:
            if asleep_frac <= ppocfg.night_lo:      # daybreak -> begin a new day of collection
                collecting = True
                cycle_start = sim.tick
        else:
            elapsed = sim.tick - cycle_start
            if elapsed >= ppocfg.min_cycle and (asleep_frac >= ppocfg.night_hi
                                                or elapsed >= ppocfg.horizon):
                # ---- NIGHT: pause the sim and run one PPO update per species ----
                rec = {"epoch": epoch, "cycle": cycle, "global_cycle": global_cycle,
                       "tick": sim.tick, "day_len": elapsed,
                       "n_sheep": stats["n_sheep"], "n_fox": stats["n_fox"]}
                msg = [f"e{epoch:3d} c{cycle:3d} | tick {sim.tick:6d} | day {elapsed:3d}t | "
                       f"sheep {stats['n_sheep']:4d} fox {stats['n_fox']:3d}"]
                for sid in (SHEEP, FOX):
                    nt = buffers[sid].n_transitions()
                    batch = buffers[sid].build_batch()
                    if batch is None:
                        msg.append(f"    {SPECIES_NAMES[sid]:5s}: no data collected")
                        continue
                    m = P.ppo_update(policies[sid], optimizers[sid], batch, ppocfg, device=DEVICE)
                    rec[f"{SPECIES_NAMES[sid]}_reward"] = float(batch["returns"].mean())
                    rec[f"{SPECIES_NAMES[sid]}_ploss"] = m["policy_loss"]
                    rec[f"{SPECIES_NAMES[sid]}_entropy"] = m["entropy"]
                    rec[f"{SPECIES_NAMES[sid]}_kl"] = m["approx_kl"]
                    rec[f"{SPECIES_NAMES[sid]}_n"] = nt
                    msg.append(f"    {SPECIES_NAMES[sid]:5s}: n={nt:6d} skip={buffers[sid].skipped:6d} "
                               f"meanR={batch['returns'].mean():+.3f} "
                               f"ploss={m['policy_loss']:+.4f} ent={m['entropy']:.3f} "
                               f"kl={m['approx_kl']:.4f} clip={m['clipfrac']:.3f}")
                    buffers[sid].clear()
                history.append(rec)
                print("\n".join(msg))

                global_cycle += 1
                if global_cycle % SAVE_EVERY == 0:
                    P.export_policy(SHEEP, policies[SHEEP], OUT_SHEEP,
                                    meta={"epoch": epoch, "cycle": cycle, "global_cycle": global_cycle})
                    P.export_policy(FOX,   policies[FOX],   OUT_FOX,
                                    meta={"epoch": epoch, "cycle": cycle, "global_cycle": global_cycle})
                    print(f"    [saved checkpoints @ epoch {epoch} cycle {cycle}]")

                collecting = False
                cycle += 1

final_pop = sim.populations if sim is not None else {}
print(f"\ndone: {N_EPOCHS} epochs, {global_cycle} cycles total, {time.time()-t0:.1f}s | "
      f"final {final_pop}")

In [ ]:
# --- export the final deployable policies (drop-in TorchScript archives) ------------------
P.export_policy(SHEEP, policies[SHEEP], OUT_SHEEP,
                meta={"epochs": N_EPOCHS, "cycles": global_cycle, "world_seed": WORLD_SEED})
P.export_policy(FOX,   policies[FOX],   OUT_FOX,
                meta={"epochs": N_EPOCHS, "cycles": global_cycle, "world_seed": WORLD_SEED})
print("wrote", OUT_SHEEP)
print("wrote", OUT_FOX)
print("\nDeploy with, e.g.:")
print(f"  venv/Scripts/python.exe run_experiment.py --ticks 3000 --world-seed {WORLD_SEED} "
      f"--seed {RUN_SEED} \\\n    --sheep-brain {OUT_SHEEP} "
      f"--fox-brain {OUT_FOX} --out runs/ppo_eval.csv --plot")

## Metrics

In [ ]:
# --- learning curves + population dynamics ------------------------------------------------
if history:
    fig, ax = plt.subplots(2, 2, figsize=(13, 8))

    for sid in (SHEEP, FOX):
        nm = SPECIES_NAMES[sid]
        rk = f"{nm}_reward"; ek = f"{nm}_entropy"; kk = f"{nm}_kl"
        # x-axis = cumulative cycle across epochs (per-epoch `cycle` resets, so it would overlay)
        xs = [h["global_cycle"] for h in history if rk in h]
        ax[0, 0].plot(xs, [h[rk] for h in history if rk in h], label=nm)
        ax[0, 1].plot(xs, [h[ek] for h in history if ek in h], label=nm)
        ax[1, 1].plot(xs, [h[kk] for h in history if kk in h], label=nm)
    ax[0, 0].set(title="mean discounted return / cycle", xlabel="cycle (cumulative)", ylabel="return")
    ax[0, 0].legend(); ax[0, 0].grid(alpha=.3)
    ax[0, 1].set(title="policy entropy (exploration)", xlabel="cycle (cumulative)"); ax[0, 1].legend(); ax[0, 1].grid(alpha=.3)
    ax[1, 1].set(title="approx KL / cycle (trust region)", xlabel="cycle (cumulative)"); ax[1, 1].legend(); ax[1, 1].grid(alpha=.3)

    # epoch boundaries on the learning curves (a new episode/sim reset happens at each)
    for a in (ax[0, 0], ax[0, 1], ax[1, 1]):
        prev = None
        for h in history:
            if h["epoch"] != prev:
                a.axvline(h["global_cycle"], color="k", ls=":", alpha=.25)
                prev = h["epoch"]

    if pop_log:
        # pop_log rows are (epoch, tick, n_sheep, n_fox); per-epoch tick resets, so plot against
        # a cumulative step index and draw a divider where each epoch (sim reset) begins.
        pl = np.array(pop_log)
        step = np.arange(len(pl))
        ax[1, 0].plot(step, pl[:, 2], label="sheep")
        ax[1, 0].plot(step, pl[:, 3], label="fox")
        epoch_starts = np.flatnonzero(np.diff(pl[:, 0]) != 0) + 1
        for s in epoch_starts:
            ax[1, 0].axvline(s, color="k", ls=":", alpha=.25)
        ax[1, 0].set(title="population over time (dotted = epoch reset)",
                     xlabel="sim step (cumulative across epochs)", ylabel="count")
        ax[1, 0].legend(); ax[1, 0].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("no cycles completed yet — run the training loop above")